<a href="https://colab.research.google.com/github/contactdilarayilmaz/asteroid-classification/blob/main/Colab%20Notebooks/03_spice_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NASA Asteroid Sınıflandırma Projesi
## Notebook 03 — SPICE ile Asteroid Yörünge Görselleştirme

**Bu notebook'ta yapacaklarımız:**
1. SpiceyPy kurulumu ve NAIF kernel'larını indirme
2. SPICE ile gezegen ve asteroid pozisyonu hesaplama
3. Apophis (99942) asteroid yörüngesini 3D olarak görselleştirme
4. 2025–2035 yörünge animasyonu (Plotly ile interaktif)
5. Apophis'in Dünya'ya mesafesini zaman içinde çizme
6. 2029 yakın geçiş anını vurgulama
7. Güneş Sistemi'nde Dünya, Mars ve Apophis yörüngelerini karşılaştırma

> **Bağlam:** Bu notebook önceki adımların *neden önemli* olduğunu görselleştirir.
> SHAP ile tehlikeli bulduğumuz asteroidi şimdi uzayda takip ediyoruz!

---
*Önceki adım:* `02_baseline_models_shap.ipynb`  
*Sonraki adım:* `04_smote_challenge.ipynb`

## 1. Kurulum — SpiceyPy ve Kernel İndirme

In [1]:
# SpiceyPy ve diğer kütüphaneleri yükle
!pip install spiceypy plotly astropy -q

print('✅ Kütüphaneler hazır!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 11.7 MB/s eta 0:00:00
✅ Kütüphaneler hazır!


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import spiceypy as spice
import warnings
warnings.filterwarnings('ignore')

# Görsel stil (önceki notebook'larla tutarlı)
plt.style.use('dark_background')
pd.set_option('display.float_format', '{:.6f}'.format)

print(f'✅ SpiceyPy versiyonu: {spice.__version__}')
print('✅ Tüm import işlemleri tamamlandı!')

✅ SpiceyPy versiyonu: 8.1.0
✅ Tüm import işlemleri tamamlandı!


## 2. SPICE Nedir? — Kısa Teorik Giriş

**SPICE** (Spacecraft Planet Instrument C-matrix Events), NASA/NAIF tarafından geliştirilmiş
gezegen bilimi altyapısıdır. Uzay görevlerinden elde edilen geometrik ve yardımcı verilerin (yörünge, yönelim, zaman sistemleri vb.) düzenlenmesi, paylaşılması ve analiz edilmesi için kullanılan gelişmiş bir bilgi sistemidir.

### Temel Kavramlar:
| Kavram | Açıklama |
|--------|----------|
| **Kernel** | SPICE'ın veri dosyaları (.bsp, .tls, .pck) |
| **SPK kernel** | Gök cisimlerinin pozisyon/hız bilgisi |
| **LSK kernel** | Artık saniye düzeltmeleri (leap seconds) |
| **PCK kernel** | Gezegen fiziksel sabitleri |
| **ET (Ephemeris Time)** | SPICE'ın iç zaman formatı |
| **J2000** | Referans koordinat sistemi (2000 yılı başı ekvatoru) |
| **ECLIPJ2000** | Güneş merkezli ekliptik koordinat sistemi |

### Neden SPICE Kullanıyoruz?
- NASA'nın resmi yörünge hesaplama aracı
- Apophis gibi gerçek asteroidlerin pozisyonlarını nanometre hassasiyetiyle verir
- Cassini, Voyager, Mars Reconnaissance Orbiter gibi tüm görevlerde kullanıldı

> **Not:** `de440.bsp` kernel'ı Güneş Sistemi gezegenlerini kapsar.
> Apophis için özel SPK kernel'ı JPL Horizons'tan indirmeliyiz.

## 3. NAIF Kernel Dosyalarını İndirme

SPICE'ın çalışması için **kernel** dosyalarına ihtiyacımız var.
Bunlar NASA sunucularından ücretsiz indirilebilir.

In [3]:
import os
import urllib.request

# Kernel dosyaları için klasör oluştur
os.makedirs('kernels', exist_ok=True)

# ─── İndirilecek kernel listesi ───────────────────────────────────────────────
KERNELS = {
    # Artık saniye tablosu (zaman dönüşümü için şart)
    'naif0012.tls': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls',
    # Güneş Sistemi gezegenlerinin efemerisleri (DE440 — modern ve hassas)
    'de440s.bsp':   'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440s.bsp',
    # Gezegen fiziksel sabitleri (yarıçap, kutup yönü, vb.)
    'pck00011.tpc': 'https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc',
}

print(' NAIF kernel dosyaları indiriliyor...')
print('   (de440s.bsp ~32MB, ilk kez 1-2 dakika sürebilir)\n')

for filename, url in KERNELS.items():
    fpath = f'kernels/{filename}'
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f'   ✅ {filename} zaten mevcut ({size_mb:.1f} MB)')
        continue
    try:
        print(f'   ⬇️  {filename} indiriliyor...')
        urllib.request.urlretrieve(url, fpath)
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f'   ✅ {filename} indirildi ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'   ❌ {filename} indirilemedi: {e}')

print('\n📦 Kernel indirme tamamlandı!')

 NAIF kernel dosyaları indiriliyor...
   (de440s.bsp ~32MB, ilk kez 1-2 dakika sürebilir)

   ⬇️  naif0012.tls indiriliyor...
   ✅ naif0012.tls indirildi (0.0 MB)
   ⬇️  de440s.bsp indiriliyor...
   ✅ de440s.bsp indirildi (31.2 MB)
   ⬇️  pck00011.tpc indiriliyor...
   ✅ pck00011.tpc indirildi (0.1 MB)

📦 Kernel indirme tamamlandı!


## 4. Apophis (99942) Kernel'ını JPL Horizons'tan İndirme

**Apophis (99942)**, 2029'da Dünya'ya çıplak gözle görülebilecek kadar yakın geçecek.
Bu özelliği projeye mükemmel bir anlatı sağlıyor.

### JPL Horizons'tan Manuel İndirme (Önerilen Yöntem):
1. https://ssd.jpl.nasa.gov/horizons/ adresine git
2. **Target Body** → `Apophis (99942)` yaz
3. **Ephemeris Type** → `SPK File` seç
4. **Time Span** → 2024-Jan-01 ile 2036-Jan-01 arası
5. **Generate Ephemeris** → Download → `wld54180.15` veya benzeri .bsp indir
6. İndirilen dosyayı `kernels/apophis.bsp` olarak kaydet

### Otomatik İndirme (Backup — Horizons API):
Aşağıdaki kod Horizons web API'sini kullanarak Apophis SPK kernel'ını otomatik indirir.

In [4]:
import os
import base64
import urllib.request

APOPHIS_SPK = 'kernels/apophis.bsp'

def download_apophis_spk():
    """JPL Horizons API üzerinden Apophis SPK kernel indirir (base64 decode ile)."""

    horizons_url = 'https://ssd.jpl.nasa.gov/api/horizons.api'

    params = (
        "format=text"
        "&COMMAND='99942'"
        "&EPHEM_TYPE='SPK'"
        "&START_TIME='2024-JAN-01'"
        "&STOP_TIME='2036-JAN-01'"
        "&CENTER='500@0'"
    )

    url = f"{horizons_url}?{params}"

    print('⏳ Horizons API\'den Apophis SPK isteği gönderiliyor...')
    print(f'   URL: {url[:80]}...')

    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            content = response.read().decode('utf-8', errors='ignore')

        # base64 bölümünü bul
        marker = 'SPK Binary Data Follows -- base64 encoded:'
        if marker not in content:
            print('   ❌ Yanıtta base64 verisi bulunamadı')
            return False

        # marker'dan sonraki kısmı al
        b64_section = content.split(marker)[1].strip()

        # Eğer sonunda ek metin varsa ($$SOF gibi), temizle
        # Base64 karakterleri: A-Z, a-z, 0-9, +, /, =, whitespace
        import re
        # Sadece base64 geçerli karakterleri tut
        b64_clean = re.sub(r'[^A-Za-z0-9+/=\s]', '', b64_section)
        b64_clean = b64_clean.strip()

        # base64 decode
        spk_binary = base64.b64decode(b64_clean)

        # DAF/SPK header kontrolü
        if spk_binary[:7] == b'DAF/SPK':
            os.makedirs('kernels', exist_ok=True)
            with open(APOPHIS_SPK, 'wb') as f:
                f.write(spk_binary)
            size_kb = len(spk_binary) / 1024
            print(f'   ✅ Apophis SPK kernel indirildi ({size_kb:.1f} KB)')
            return True
        else:
            print(f'   ⚠️  Decode edilen veri SPK formatında değil (header: {spk_binary[:10]})')
            return False

    except Exception as e:
        print(f'   ❌ API isteği başarısız: {e}')
        import traceback
        traceback.print_exc()
        return False


if __name__ == '__main__':
    if os.path.exists(APOPHIS_SPK):
        size_kb = os.path.getsize(APOPHIS_SPK) / 1024
        print(f'✅ {APOPHIS_SPK} zaten mevcut ({size_kb:.1f} KB)')
    else:
        success = download_apophis_spk()
        if not success:
            print('\n💡 Manuel İndirme Adımları:')
            print('   1. https://ssd.jpl.nasa.gov/horizons/app.html adresine git')
            print('   2. Target Body: 99942 Apophis')
            print('   3. Ephemeris Type: Small-Body SPK File')
            print('   4. Time Span: 2024-Jan-01 to 2036-Jan-01')
            print('   5. Generate SPK → İndir')
            print(f'   6. İndirilen dosyayı {APOPHIS_SPK} olarak kaydet')

⏳ Horizons API'den Apophis SPK isteği gönderiliyor...
   URL: https://ssd.jpl.nasa.gov/api/horizons.api?format=text&COMMAND='99942'&EPHEM_TYPE...
   ✅ Apophis SPK kernel indirildi (310.0 KB)


## 5. SPICE Kernel'larını Yükle ve İlk Test

In [5]:
# Tüm kernel'ları SPICE'a yükle
print('🔧 SPICE kernel\'ları yükleniyor...')

# Önceki oturumda yüklenmiş olabilir — temizle
try:
    spice.kclear()
except:
    pass

# Temel kernel'lar
LOADED_KERNELS = []
for fname in ['naif0012.tls', 'de440s.bsp', 'pck00011.tpc']:
    fpath = f'kernels/{fname}'
    if os.path.exists(fpath):
        spice.furnsh(fpath)
        LOADED_KERNELS.append(fname)
        print(f'   ✅ {fname} yüklendi')
    else:
        print(f'   ⚠️  {fname} bulunamadı — lütfen indirme adımını tekrar çalıştır')

# Apophis kernel'ı (varsa)
APOPHIS_KERNEL_LOADED = False
if os.path.exists('kernels/apophis.bsp'):
    spice.furnsh('kernels/apophis.bsp')
    LOADED_KERNELS.append('apophis.bsp')
    APOPHIS_KERNEL_LOADED = True
    print('   ✅ apophis.bsp yüklendi')
else:
    print('   ⚠️  apophis.bsp bulunamadı — orbital parametre modu kullanılacak')

print(f'\n📊 Yüklü kernel sayısı: {len(LOADED_KERNELS)}')

# ─── Basit test: Dünya'nın pozisyonu ─────────────────────────────────────────
# str2et: İnsan okunabilir tarihi Ephemeris Time'a çevirir
et_test = spice.str2et('2025-01-01')
earth_pos, lt = spice.spkpos('EARTH', et_test, 'J2000', 'NONE', 'SUN')

print(f'\n🌍 Test — Dünya\'nın 2025-01-01 pozisyonu (J2000, güneş merkezli):')
print(f'   X = {earth_pos[0]/1.496e8:.4f} AU')
print(f'   Y = {earth_pos[1]/1.496e8:.4f} AU')
print(f'   Z = {earth_pos[2]/1.496e8:.4f} AU')
print(f'   Güneş\'e mesafe: {np.linalg.norm(earth_pos)/1.496e8:.4f} AU  (≈1 AU beklenir)')
print('\n✅ SPICE başarıyla çalışıyor!')

🔧 SPICE kernel'ları yükleniyor...
   ✅ naif0012.tls yüklendi
   ✅ de440s.bsp yüklendi
   ✅ pck00011.tpc yüklendi
   ✅ apophis.bsp yüklendi

📊 Yüklü kernel sayısı: 4

🌍 Test — Dünya'nın 2025-01-01 pozisyonu (J2000, güneş merkezli):
   X = -0.1787 AU
   Y = 0.8872 AU
   Z = 0.3846 AU
   Güneş'e mesafe: 0.9833 AU  (≈1 AU beklenir)

✅ SPICE başarıyla çalışıyor!


In [6]:
# SPK dosyasının içindeki objeleri listele
objects = spice.spkobj('kernels/apophis.bsp')

for obj in objects:
    # Her objenin hangi zaman aralığını kapsadığını bul
    coverage = spice.spkcov('kernels/apophis.bsp', obj)
    print(f"Cisim ID: {obj}")
    for i in range(spice.wncard(coverage)):
        begin, end = spice.wnfetd(coverage, i)
        print(f"  Aralık: {spice.et2utc(begin, 'C', 1)} --> {spice.et2utc(end, 'C', 1)}")

Cisim ID: 20099942
  Aralık: 2023 DEC 31 23:58:50.8 --> 2035 DEC 31 23:58:50.8


## 6. Dünya ve Apophis Pozisyonlarını Hesapla (2025–2035)

SPICE'ın çekirdeği: belirli bir zaman diliminde gezegen/asteroid pozisyonlarını
Ephemeris kernel'larından okuyarak döndürür.

In [9]:
# ─── Zaman aralığı tanımla ────────────────────────────────────────────────────
# 2025 başından 2036 başına, 7 günde bir
date_start = '2025-01-01'
date_end   = '2036-01-01'
step_days  = 7  # Gün

# Tarih listesi oluştur
import datetime
dates = []
current = datetime.date(2025, 1, 1)
end_date = datetime.date(2036, 1, 1)
while current <= end_date:
    dates.append(current.strftime('%Y-%m-%d'))
    current += datetime.timedelta(days=step_days)

# Epoch zamanlarına (ET) çevir
et_times = [spice.str2et(d) for d in dates]
n_steps  = len(et_times)
print(f'📅 Zaman adımı sayısı: {n_steps} ({step_days} günde bir, ~11 yıl)')

# ─── AU dönüşüm sabiti ────────────────────────────────────────────────────────
AU = 1.496e8  # km/AU

# ─── Dünya pozisyonları ───────────────────────────────────────────────────────
print('\n⏳ Dünya pozisyonları hesaplanıyor...')
earth_positions = np.array([
    spice.spkpos('EARTH', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
    for et in et_times
]) / AU  # km → AU

print(f'   ✅ Dünya: {earth_positions.shape} pozisyon vektörü')

# ─── Mars pozisyonları (karşılaştırma için) ───────────────────────────────────
print('⏳ Mars pozisyonları hesaplanıyor...')
mars_positions = np.array([
    spice.spkpos('MARS BARYCENTER', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
    for et in et_times
]) / AU

print(f'   ✅ Mars: {mars_positions.shape} pozisyon vektörü')

# ─── Apophis pozisyonları ─────────────────────────────────────────────────────
if APOPHIS_KERNEL_LOADED:
    print('⏳ Apophis pozisyonları hesaplanıyor (SPK kernel)...')
    try:
        apophis_positions = np.array([
            spice.spkpos('20099942', et, 'ECLIPJ2000', 'NONE', 'SUN')[0]
            for et in et_times
        ]) / AU
        print(f'   ✅ Apophis: {apophis_positions.shape} pozisyon vektörü (SPK kernel)')
        SPICE_APOPHIS = True
    except Exception as e:
        print(f'   ⚠️  SPK hatası: {e}')
        SPICE_APOPHIS = False
else:
    SPICE_APOPHIS = False

print('\n✅ Tüm pozisyon verileri hazır!')
print(f'   Zaman aralığı: {dates[0]} → {dates[-1]}')

📅 Zaman adımı sayısı: 574 (7 günde bir, ~11 yıl)

⏳ Dünya pozisyonları hesaplanıyor...
   ✅ Dünya: (574, 3) pozisyon vektörü
⏳ Mars pozisyonları hesaplanıyor...
   ✅ Mars: (574, 3) pozisyon vektörü
⏳ Apophis pozisyonları hesaplanıyor (SPK kernel)...
   ✅ Apophis: (574, 3) pozisyon vektörü (SPK kernel)

✅ Tüm pozisyon verileri hazır!
   Zaman aralığı: 2025-01-01 → 2035-12-26


## 7. Apophis'in Dünya'ya Mesafesini Hesapla

In [ ]:
from scipy.signal import argrelmin

# --- Apophis'in Dunya'ya mesafesi (AU) ---
apophis_to_earth = np.linalg.norm(apophis_positions - earth_positions, axis=1)
dates_arr = np.array(dates)

# --- Istatistikler ---
global_min_dist = apophis_to_earth.min()
global_max_dist = apophis_to_earth.max()
print('Apophis - Dunya Mesafe Istatistikleri (AU):')
print(f'   Min : {global_min_dist:.4f} AU  ({global_min_dist*1.496e8:.0f} km)')
print(f'   Max : {global_max_dist:.4f} AU  ({global_max_dist*1.496e8:.0f} km)')
print(f'   Ort : {apophis_to_earth.mean():.4f} AU')

# --- PHA karsilastirmasi (DINAMIK - hesaplanan mesafeye gore) ---
PHA_THRESHOLD_AU = 0.05   # NASA PHA kriteri: MOID <= 0.05 AU
is_pha_candidate = global_min_dist <= PHA_THRESHOLD_AU
if is_pha_candidate:
    pha_status = f'KARSILANIYOR => Bu cisim PHA sinifina girer (min={global_min_dist:.5f} AU <= {PHA_THRESHOLD_AU} AU)'
else:
    pha_status = f'KARSILANMIYOR => Bu cisim PHA sinifina girmez (min={global_min_dist:.5f} AU > {PHA_THRESHOLD_AU} AU)'
print(f'\n   PHA kriteri (MOID <= {PHA_THRESHOLD_AU} AU): {pha_status}')

# --- Global minimum: tum zaman diliminde en yakin an (otomatik tespit) ---
global_min_idx = np.argmin(apophis_to_earth)
print(f'\nGlobal Minimum (Tum Donem Icinde En Yakin An):')
print(f'   Tarih    : {dates_arr[global_min_idx]}')
print(f'   Mesafe   : {apophis_to_earth[global_min_idx]:.5f} AU  ({apophis_to_earth[global_min_idx]*1.496e8:.0f} km)')
print(f'   Referans : Ay Yorungesi ~0.00257 AU (384,400 km)')

# --- Tum yerel minimumlar: scipy.signal.argrelmin ile her yakin gecis ani ---
# order=3 => her iki yanda 3 noktayla karsilastirilarak yerel min belirlenir
local_min_indices = argrelmin(apophis_to_earth, order=3)[0]

print(f'\nTum Yerel Minimumlar ({len(local_min_indices)} yakin gecis tespit edildi):')
header = f'   {"Sira":<5} {"Tarih":<14} {"Mesafe (AU)":<14} {"Mesafe (km)":<15} {"PHA?"}'
print(header)
print('   ' + '-'*65)
for rank, idx in enumerate(sorted(local_min_indices, key=lambda i: apophis_to_earth[i]), 1):
    d    = apophis_to_earth[idx]
    flag = 'EVET (PHA!)' if d <= PHA_THRESHOLD_AU else 'hayir'
    print(f'   {rank:<5} {dates_arr[idx]:<14} {d:<14.5f} {d*1.496e8:<15.0f} {flag}')

## 8. Apophis'in Dünya'ya Mesafesi — Zaman Grafiği

In [ ]:
import datetime as dt

fig, ax = plt.subplots(figsize=(14, 6))

# Tarih listesini datetime objesine cevir
date_objects = [dt.datetime.strptime(d, '%Y-%m-%d') for d in dates]

# Ana egri
ax.plot(date_objects, apophis_to_earth, color='#4ECDC4', linewidth=2, label='Apophis-Dunya Mesafesi')
ax.fill_between(date_objects, apophis_to_earth, alpha=0.15, color='#4ECDC4')

# PHA esik cizgisi (0.05 AU)
ax.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5, alpha=0.8,
           label='PHA Esigi (0.05 AU)')

# Ay yorunge mesafesi referansi
ax.axhline(y=0.00257, color='#FFEAA7', linestyle=':', linewidth=1.5, alpha=0.8,
           label='Ay Yorungesi (0.00257 AU)')

# --- En yakin gecisi vurgula: global minimum otomatik tespit ---
global_min_idx  = np.argmin(apophis_to_earth)
global_min_dist_val = apophis_to_earth[global_min_idx]
global_min_date = date_objects[global_min_idx]

# Annotation ofseti: minimum tarihinden ~18 ay once
offset_date = global_min_date - dt.timedelta(days=548)
ax.annotate(
    f'En Yakin Gecis\n{global_min_dist_val:.4f} AU\n({global_min_dist_val*1.496e8:,.0f} km)',
    xy=(global_min_date, global_min_dist_val),
    xytext=(offset_date, global_min_dist_val + 0.15),
    fontsize=10, color='#FF6B6B', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#FF6B6B', lw=2),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e', edgecolor='#FF6B6B')
)

# Yil dikey cizgileri: dates verisiyle otomatik yil aralik
start_year = int(dates[0][:4])
end_year   = int(dates[-1][:4]) + 1
for year in range(start_year, end_year + 1):
    ax.axvline(x=dt.datetime(year, 1, 1), color='gray', alpha=0.2, linewidth=0.8)

ax.set_xlabel('Tarih', fontsize=12)
ax.set_ylabel('Apophis-Dunya Mesafesi (AU)', fontsize=12)
title_start = dates[0][:4]
title_end   = dates[-1][:4]
ax.set_title(f'Apophis (99942) - Dunya\'ya Mesafe ({title_start}-{title_end})\n'
             'Yakin gecis tarihleri ve PHA esigi', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('apophis_distance_timeline.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()
print('apophis_distance_timeline.png kaydedildi')

## 9. 2D Yörünge Haritası — Güneş Sistemi'ne Genel Bakış

In [ ]:
# --- 2D gorunum: Ekliptik duzlem (x-y) ---
fig, ax = plt.subplots(figsize=(11, 11))
ax.set_facecolor('#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')

# Gunes
sun = plt.Circle((0, 0), 0.04, color='#FFD93D', zorder=5, label='Gunes')
ax.add_patch(sun)

# Dunya yorungesi
ax.plot(earth_positions[:, 0], earth_positions[:, 1],
        color='#4ECDC4', linewidth=1.5, alpha=0.7, label='Dunya Yorungesi')
ax.scatter(earth_positions[-1, 0], earth_positions[-1, 1],
           color='#4ECDC4', s=80, zorder=6, marker='o')

# Mars yorungesi
ax.plot(mars_positions[:, 0], mars_positions[:, 1],
        color='#FF6B6B', linewidth=1.5, alpha=0.5, label='Mars Yorungesi')
ax.scatter(mars_positions[-1, 0], mars_positions[-1, 1],
           color='#FF6B6B', s=80, zorder=6, marker='o')

# Apophis yorungesi
ax.plot(apophis_positions[:, 0], apophis_positions[:, 1],
        color='#A8E6CF', linewidth=2, alpha=0.9, label='Apophis Yorungesi')

# --- En yakin gecis noktasini dinamik olarak bul ve vurgula ---
global_min_idx      = np.argmin(apophis_to_earth)
global_min_dist_val = apophis_to_earth[global_min_idx]
global_min_date_str = dates_arr[global_min_idx]   # 'YYYY-MM-DD' string

ax.scatter(
    apophis_positions[global_min_idx, 0],
    apophis_positions[global_min_idx, 1],
    color='#FFEAA7', s=200, zorder=7, marker='*',
    label=f'En Yakin Gecis ({global_min_date_str})'
)
ax.annotate(
    f'En Yakin Gecis\n{global_min_date_str}',
    xy=(apophis_positions[global_min_idx, 0],
        apophis_positions[global_min_idx, 1]),
    xytext=(apophis_positions[global_min_idx, 0] + 0.15,
            apophis_positions[global_min_idx, 1] + 0.15),
    color='#FFEAA7', fontsize=9, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#FFEAA7')
)

# 1 AU referans cemberi
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), color='gray', alpha=0.2, linewidth=0.8,
        linestyle=':', label='1 AU referansi')

# Zaman araligini baslikta dinamik goster
start_yr = dates_arr[0][:4]
end_yr   = dates_arr[-1][:4]

ax.set_xlabel('X (AU) - Ekliptik Koordinat', fontsize=11)
ax.set_ylabel('Y (AU) - Ekliptik Koordinat', fontsize=11)
ax.set_title(f'Ic Gunes Sistemi - Ekliptik Duzlem Gorunumu ({start_yr}-{end_yr})\n'
             'Apophis, Dunya ve Mars yorungeleri', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, facecolor='#1a1a2e', edgecolor='gray')
ax.set_aspect('equal')
ax.tick_params(colors='white')
ax.grid(alpha=0.15)

lim = 1.8
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

plt.tight_layout()
plt.savefig('solar_system_2d.png', dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()
print('solar_system_2d.png kaydedildi')


## 10. İnteraktif 3D Yörünge Görselleştirmesi (Plotly)

In [ ]:
fig_3d = go.Figure()

# --- Gunes ---
fig_3d.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=12, color='#FFD93D', symbol='circle'),
    name='Gunes',
    hovertext='Gunes (Koordinat Merkezi)'
))

# --- Dunya yorungesi ---
fig_3d.add_trace(go.Scatter3d(
    x=earth_positions[:, 0],
    y=earth_positions[:, 1],
    z=earth_positions[:, 2],
    mode='lines',
    line=dict(color='#4ECDC4', width=3),
    name='Dunya Yorungesi',
    text=dates,
    hovertemplate='%{text}<br>X:%{x:.3f} Y:%{y:.3f} Z:%{z:.3f} AU<extra>Dunya</extra>'
))

# --- Mars yorungesi ---
fig_3d.add_trace(go.Scatter3d(
    x=mars_positions[:, 0],
    y=mars_positions[:, 1],
    z=mars_positions[:, 2],
    mode='lines',
    line=dict(color='#FF6B6B', width=2, dash='dash'),
    name='Mars Yorungesi',
    opacity=0.5
))

# --- Apophis yorungesi (renk = Dunya'ya mesafe) ---
fig_3d.add_trace(go.Scatter3d(
    x=apophis_positions[:, 0],
    y=apophis_positions[:, 1],
    z=apophis_positions[:, 2],
    mode='lines+markers',
    line=dict(color='#A8E6CF', width=4),
    marker=dict(
        size=3,
        color=apophis_to_earth,
        colorscale='RdYlGn_r',
        colorbar=dict(title='Dunya<br>Mesafe (AU)', thickness=15, x=1.0),
        showscale=True,
        cmin=0,
        cmax=apophis_to_earth.max()
    ),
    name='Apophis (99942)',
    text=[f'{d} | {dist:.4f} AU' for d, dist in zip(dates, apophis_to_earth)],
    hovertemplate='%{text}<extra>Apophis</extra>'
))

# --- En yakin gecis noktasi: dinamik global minimum ---
global_min_idx      = np.argmin(apophis_to_earth)
global_min_dist_val = apophis_to_earth[global_min_idx]
global_min_date_str = dates_arr[global_min_idx]

fig_3d.add_trace(go.Scatter3d(
    x=[apophis_positions[global_min_idx, 0]],
    y=[apophis_positions[global_min_idx, 1]],
    z=[apophis_positions[global_min_idx, 2]],
    mode='markers+text',
    marker=dict(size=14, color='#FFEAA7', symbol='diamond',
                line=dict(color='white', width=2)),
    text=['En Yakin Gecis'],
    textposition='top center',
    name=f'En Yakin Gecis ({global_min_date_str})',
    hovertext=(
        f'{global_min_date_str}: {global_min_dist_val:.5f} AU '
        f'({global_min_dist_val*1.496e8:,.0f} km)'
    )
))

# --- Layout ---
start_yr = dates_arr[0][:4]
end_yr   = dates_arr[-1][:4]

fig_3d.update_layout(
    title=dict(
        text=(f'Apophis (99942) - 3D Yorunge Gorsellestirmesi ({start_yr}-{end_yr})<br>'
              '<sub>Noktalarin rengi Dunya ya olan mesafeyi gosteriyor (kirmizi=yakin)</sub>'),
        x=0.5,
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='X (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        yaxis=dict(title='Y (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        zaxis=dict(title='Z (AU)', backgroundcolor='#0a0a1a', gridcolor='#333'),
        bgcolor='#0a0a1a',
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0.5)'),
    width=900,
    height=700,
)

fig_3d.write_html('apophis_3d_orbit.html')
fig_3d.show()
print('apophis_3d_orbit.html kaydedildi')


## 11. Apophis Yörünge Animasyonu (Plotly)

In [14]:
# Animasyon: Her frame bir zaman dilimini gösterir
# Performans için her 4. noktayı al
step = 4
anim_dates = dates[::step]
anim_earth  = earth_positions[::step]
anim_apophis = apophis_positions[::step]
anim_distances = apophis_to_earth[::step]
n_frames = len(anim_dates)

print(f'🎬 Animasyon frame sayısı: {n_frames} ({step} adımda bir)')

# ─── Frame verileri ───────────────────────────────────────────────────────────
frames = []
for i in range(n_frames):
    frame_data = [
        # Apophis izi (0'dan i'ye kadar)
        go.Scatter3d(
            x=anim_apophis[:i+1, 0],
            y=anim_apophis[:i+1, 1],
            z=anim_apophis[:i+1, 2],
            mode='lines+markers',
            line=dict(color='#A8E6CF', width=3),
            marker=dict(size=[2]*(i) + [8], color='#A8E6CF'),
        ),
        # Dünya izi
        go.Scatter3d(
            x=anim_earth[:i+1, 0],
            y=anim_earth[:i+1, 1],
            z=anim_earth[:i+1, 2],
            mode='lines+markers',
            line=dict(color='#4ECDC4', width=2),
            marker=dict(size=[2]*(i) + [8], color='#4ECDC4'),
        ),
    ]
    frames.append(go.Frame(data=frame_data, name=anim_dates[i]))

# ─── Başlangıç şekli (ilk frame) ─────────────────────────────────────────────
fig_anim = go.Figure(
    data=[
        go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                     marker=dict(size=10, color='#FFD93D'), name='Güneş'),
        go.Scatter3d(x=anim_apophis[:1, 0], y=anim_apophis[:1, 1], z=anim_apophis[:1, 2],
                     mode='markers', marker=dict(size=8, color='#A8E6CF'), name='Apophis'),
        go.Scatter3d(x=anim_earth[:1, 0], y=anim_earth[:1, 1], z=anim_earth[:1, 2],
                     mode='markers', marker=dict(size=8, color='#4ECDC4'), name='Dünya'),
    ],
    frames=frames
)

fig_anim.update_layout(
    title=dict(text='🎬 Apophis Animasyonu — 2025–2035', x=0.5, font=dict(size=15)),
    scene=dict(
        xaxis=dict(title='X (AU)', range=[-2, 2], backgroundcolor='#0a0a1a'),
        yaxis=dict(title='Y (AU)', range=[-2, 2], backgroundcolor='#0a0a1a'),
        zaxis=dict(title='Z (AU)', range=[-0.5, 0.5], backgroundcolor='#0a0a1a'),
        bgcolor='#0a0a1a',
    ),
    paper_bgcolor='#1a1a2e',
    font=dict(color='white'),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=1.05, x=0.5,
        buttons=[
            dict(label='▶ Oynat',
                 method='animate',
                 args=[None, dict(frame=dict(duration=80, redraw=True),
                                  fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Durdur',
                 method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode='immediate')])
        ]
    )],
    sliders=[dict(
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                    method='animate', label=f.name[::30]) for f in frames],
        active=0, y=0, len=1.0, x=0, currentvalue=dict(prefix='Tarih: ', font=dict(size=12))
    )],
    width=900, height=700,
)

fig_anim.write_html('apophis_animation.html')
fig_anim.show()
print('💾 apophis_animation.html kaydedildi')

🎬 Animasyon frame sayısı: 144 (4 adımda bir)


💾 apophis_animation.html kaydedildi


## 12. Tehlike Penceresi Analizi — Yakın Geçiş Yakınlaştırması

In [ ]:
# --- En yakin gecis etrafinda dinamik tehlike penceresi ---
# Global minimum otomatik tespit
global_min_idx  = np.argmin(apophis_to_earth)
global_min_dist_val = apophis_to_earth[global_min_idx]
global_min_date = date_objects[global_min_idx]

# Minimum etrafinda +/- 18 ay dinamik pencere
window_start = global_min_date - dt.timedelta(days=548)
window_end   = global_min_date + dt.timedelta(days=548)

mask_critical = np.array([
    window_start <= dt.datetime.strptime(d, '%Y-%m-%d') <= window_end
    for d in dates
])
dates_critical = np.array(date_objects)[mask_critical]
dist_critical  = apophis_to_earth[mask_critical]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Panel 1: Tum donem ---
ax1 = axes[0]
ax1.plot(date_objects, apophis_to_earth, color='#4ECDC4', linewidth=1.5)
ax1.fill_between(date_objects, 0, apophis_to_earth,
                 where=apophis_to_earth < 0.05,
                 alpha=0.3, color='#FF6B6B', label='PHA tehlike bolgesi')
ax1.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5, label='PHA Esigi (0.05 AU)')

# axvline: sabit tarih degil, hesaplanan global minimum tarihi
ax1.axvline(x=global_min_date, color='#FFEAA7', linestyle=':', linewidth=1.5,
            label=f'Hesaplanan En Yakin Gecis: {global_min_date.strftime("%d %b %Y")}')

start_yr = int(dates[0][:4])
end_yr   = int(dates[-1][:4])
ax1.set_title(f'Apophis-Dunya Mesafesi - Tam Donem ({start_yr}-{end_yr})', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mesafe (AU)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.2)
ax1.set_ylim(0)

# Zoom alanini kutucukla goster — dinamik pencere koordinatlari
from matplotlib.patches import Rectangle
rect_width = window_end - window_start
ax1.add_patch(Rectangle(
    (window_start, 0), rect_width, 0.6,
    linewidth=2, edgecolor='#FFEAA7', facecolor='none', linestyle='--', alpha=0.7
))
label_pos = window_start + dt.timedelta(days=30)
ax1.text(label_pos, 0.55, 'Yakinlastirilan bolge',
         color='#FFEAA7', fontsize=9)

# --- Panel 2: Yakinlastirma ---
ax2 = axes[1]
ax2.plot(dates_critical, dist_critical, color='#FFEAA7', linewidth=2.5)
ax2.fill_between(dates_critical, 0, dist_critical,
                 where=dist_critical < 0.05,
                 alpha=0.4, color='#FF6B6B')
ax2.fill_between(dates_critical, 0, dist_critical,
                 where=dist_critical >= 0.05,
                 alpha=0.15, color='#4ECDC4')

# Esik cizgileri
ax2.axhline(y=0.05, color='#FF6B6B', linestyle='--', linewidth=1.5,
            label='PHA Esigi (0.05 AU)')
ax2.axhline(y=0.00257, color='#96CEB4', linestyle=':', linewidth=1.5,
            label='Ay Yorungesi (0.00257 AU)')

# Minimum noktayi isaretleme: pencere icerisindeki min
if mask_critical.sum() > 0:
    idx_min = np.argmin(dist_critical)
    ax2.scatter(dates_critical[idx_min], dist_critical[idx_min],
                color='#FF4444', s=200, zorder=5,
                label=f'En yakin: {dist_critical[idx_min]:.4f} AU')
    ax2.annotate(
        f'Min: {dist_critical[idx_min]:.5f} AU\n({dist_critical[idx_min]*1.496e8:,.0f} km)',
        xy=(dates_critical[idx_min], dist_critical[idx_min]),
        xytext=(dates_critical[idx_min], dist_critical[idx_min] + 0.04),
        fontsize=9, color='#FF4444', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#FF4444')
    )

window_label = f'{window_start.strftime("%Y-%m")} / {window_end.strftime("%Y-%m")}'
ax2.set_title(f'Yakinlastirma: {window_label} — Apophis Tehlike Penceresi', fontsize=12, fontweight='bold')
ax2.set_xlabel('Tarih')
ax2.set_ylabel('Mesafe (AU)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)
ax2.set_ylim(0)

plt.suptitle('Apophis Tehlike Analizi', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('apophis_danger_window.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()
print('apophis_danger_window.png kaydedildi')


## 13. Önemli Tarihler ve Veriler — Özet Tablosu

In [ ]:
# --- Ozet istatistikler ---
# PHA esigi altinda gecen sureyi hesapla
PHA_THRESHOLD_AU = 0.05
pha_below  = apophis_to_earth < PHA_THRESHOLD_AU
days_below = pha_below.sum() * step_days  # yaklasik gun sayisi

# Yillari veriden otomatik cikar (hardcode aralik yok)
years = sorted(set(int(d[:4]) for d in dates))

# Yillik minimum mesafe tablosu
print('YILLIK MINIMUM MESAFE TABLOSU')
print('='*65)
print(f'{"Yil":<6} {"Min Mesafe (AU)":<20} {"Min Mesafe (km)":<22} {"Tarih":<14} PHA?')
print('-'*65)

annual_summary = []
for year in years:   # veriden turetilen yil listesi
    mask_year = np.array([d.startswith(str(year)) for d in dates])
    if mask_year.sum() == 0:
        continue
    dates_year = np.array(dates)[mask_year]
    dist_year  = apophis_to_earth[mask_year]
    idx_min    = np.argmin(dist_year)
    min_dist   = dist_year[idx_min]
    min_date   = dates_year[idx_min]
    is_pha     = 'EVET (PHA!)' if min_dist < PHA_THRESHOLD_AU else '-'
    print(f'{year:<6} {min_dist:<20.5f} {min_dist*1.496e8:<22,.0f} {min_date:<14} {is_pha}')
    annual_summary.append({
        'Yil': year, 'Min AU': min_dist,
        'Min km': min_dist * 1.496e8, 'Tarih': min_date,
        'PHA': is_pha
    })

print('='*65)
print(f'PHA esigi: < {PHA_THRESHOLD_AU} AU (NASA kriteri)')
print(f'PHA esigi altinda gecen yaklasik sure: ~{days_below} gun')

# Pandas tablosuna da cevir
summary_df = pd.DataFrame(annual_summary)
print()
print(summary_df.to_string(index=False))

## 14. Drive'a Kaydet

In [17]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

DRIVE_PATH = '/content/drive/MyDrive/nasa_asteroid/'
os.makedirs(DRIVE_PATH, exist_ok=True)

# Görselleri Drive'a kopyala
outputs = [
    'apophis_distance_timeline.png',
    'solar_system_2d.png',
    'apophis_3d_orbit.html',
    'apophis_animation.html',
    'apophis_danger_window.png',
]

print('💾 Dosyalar Drive\'a kaydediliyor...')
for fname in outputs:
    if os.path.exists(fname):
        shutil.copy(fname, DRIVE_PATH + fname)
        print(f'   ✅ {fname}')
    else:
        print(f'   ⚠️  {fname} bulunamadı')

# Pozisyon verilerini numpy olarak kaydet
np.save('apophis_positions.npy', apophis_positions)
np.save('earth_positions.npy', earth_positions)
np.save('mars_positions.npy', mars_positions)
np.save('distances_au.npy', apophis_to_earth)

for fname in ['apophis_positions.npy', 'earth_positions.npy',
              'mars_positions.npy', 'distances_au.npy']:
    shutil.copy(fname, DRIVE_PATH + fname)
    print(f'   ✅ {fname}')

print(f'\n🚀 Notebook 03 tamamlandı!')
print(f'   Sıradaki adım: 04_smote_challenge.ipynb')

Mounted at /content/drive
💾 Dosyalar Drive'a kaydediliyor...
   ✅ apophis_distance_timeline.png
   ✅ solar_system_2d.png
   ✅ apophis_3d_orbit.html
   ✅ apophis_animation.html
   ✅ apophis_danger_window.png
   ✅ apophis_positions.npy
   ✅ earth_positions.npy
   ✅ mars_positions.npy
   ✅ distances_au.npy

🚀 Notebook 03 tamamlandı!
   Sıradaki adım: 04_smote_challenge.ipynb


## 📋 Bu Notebook'tan Çıkarımlar

### SPICE ile Yapılanlar
| Görev | Araç | Çıktı |
|-------|------|-------|
| Gezegen pozisyonu | `spice.spkpos()` | x, y, z koordinatları (AU) |
| Zaman dönüşümü | `spice.str2et()` | Ephemeris Time |
| Kernel yükleme | `spice.furnsh()` | de440s.bsp, naif0012.tls |
| Apophis yörüngesi | SPK kernel / Kepler | 11 yıllık pozisyon dizisi |

### Önemli Bulgular
- **2029 Yakın Geçiş**: Apophis, Dünya'ya tarihsel olarak yakın geçecek
- **PHA Kriteri**: MOID ≤ 0.05 AU — Apophis bu eşiğin altına iniyor
- **Ay Yörüngesi Referansı**: 0.00257 AU ≈ 384,400 km
- **Görsel Doğrulama**: ML modelimizin tehlikeli dediği asteroidin neden tehlikeli olduğunu gösterdik

### Raporun İçin Not Et
- `apophis_3d_orbit.html` interaktif — sunumda tarayıcıda göster
- 2D ekliptik haritası: Apophis'in Dünya yörüngesiyle kesiştiğini gösteriyor
- Kepler fallback vs SPICE farkını tartış (kernel yoksa nasıl devam edilir?)
- SPICE'ın NASA görevlerinde kullanımından bahset (bağlam için)

### ⏭️ Sıradaki Notebook
`04_smote_challenge.ipynb` → SMOTE ile sınıf dengesizliğini çözme